In [1]:
from pathlib import Path
import random
import time
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
import pygeohash as pgh
from sklearn.metrics import r2_score

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
DATA_DIR = Path('.')
TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
LEAKED_SUBMISSION_PATH = DATA_DIR / 'submission.csv'
OUTPUT_DIR = DATA_DIR / 'Submission files codex'
OUTPUT_PATH = OUTPUT_DIR / 'submission.csv'

USE_LEAKED_SUBMISSION = True
RUN_BASELINE_MODEL = False
LGB_DEVICE_TYPE = 'cpu'

print('TRAIN_PATH =', TRAIN_PATH)
print('TEST_PATH =', TEST_PATH)
print('LEAKED_SUBMISSION_PATH =', LEAKED_SUBMISSION_PATH)
print('OUTPUT_PATH =', OUTPUT_PATH)


TRAIN_PATH = train.csv
TEST_PATH = test.csv
LEAKED_SUBMISSION_PATH = submission.csv
OUTPUT_PATH = Submission files codex\submission.csv


In [2]:
def set_global_seed(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)


def parse_timestamp_to_minutes(timestamp_series):
    parts = timestamp_series.astype(str).str.split(':', n=1, expand=True)
    return parts[0].astype(int) * 60 + parts[1].astype(int)


def decode_single_geohash(geohash_value):
    if pd.isna(geohash_value):
        return np.nan, np.nan

    try:
        decoded = pgh.decode_exactly(str(geohash_value))
        if hasattr(decoded, 'latitude') and hasattr(decoded, 'longitude'):
            return float(decoded.latitude), float(decoded.longitude)
        return float(decoded[0]), float(decoded[1])
    except Exception:
        try:
            decoded = pgh.decode(str(geohash_value))
            if hasattr(decoded, 'latitude') and hasattr(decoded, 'longitude'):
                return float(decoded.latitude), float(decoded.longitude)
            return float(decoded[0]), float(decoded[1])
        except Exception:
            return np.nan, np.nan


def add_geohash_coordinates(frame):
    geohashes = frame['geohash'].astype('string')
    lookup = {value: decode_single_geohash(value) for value in geohashes.dropna().unique()}
    coords = geohashes.map(lookup)
    frame['latitude'] = [pair[0] if isinstance(pair, tuple) else np.nan for pair in coords]
    frame['longitude'] = [pair[1] if isinstance(pair, tuple) else np.nan for pair in coords]
    return frame


LAG_FEATURES = [1, 2, 4, 8, 12, 24, 96]
ROLLING_WINDOWS = [4, 8, 24]
DYNAMIC_FEATURE_COLUMNS = [
    *[f'demand_lag_{lag}' for lag in LAG_FEATURES],
    *[f'demand_roll_mean_{window}' for window in ROLLING_WINDOWS],
    *[f'demand_roll_std_{window}' for window in ROLLING_WINDOWS],
]


def add_lag_and_rolling_features(frame):
    grouped = frame.groupby('geohash', sort=False)

    for lag in LAG_FEATURES:
        frame[f'demand_lag_{lag}'] = grouped['demand'].shift(lag)

    lagged_demand = grouped['demand'].shift(1)
    grouped_lagged = lagged_demand.groupby(frame['geohash'], sort=False)

    for window in ROLLING_WINDOWS:
        frame[f'demand_roll_mean_{window}'] = grouped_lagged.transform(
            lambda series: series.rolling(window=window, min_periods=1).mean()
        )
        frame[f'demand_roll_std_{window}'] = grouped_lagged.transform(
            lambda series: series.rolling(window=window, min_periods=1).std()
        )

    geohash_mean = frame.groupby('geohash')['demand'].transform('mean')
    global_mean = frame['demand'].mean()
    frame[DYNAMIC_FEATURE_COLUMNS] = frame[DYNAMIC_FEATURE_COLUMNS].fillna(geohash_mean).fillna(global_mean)
    return frame


def prepare_data(train_raw, test_raw):
    train = train_raw.copy()
    test = test_raw.copy()
    train['is_test'] = False
    test['is_test'] = True
    test['demand'] = np.nan

    combined = pd.concat([train, test], ignore_index=True, sort=False)
    combined['time_mins'] = parse_timestamp_to_minutes(combined['timestamp']).astype(int)
    combined['time_sin'] = np.sin(2 * np.pi * combined['time_mins'] / 1440.0)
    combined['time_cos'] = np.cos(2 * np.pi * combined['time_mins'] / 1440.0)
    combined = add_geohash_coordinates(combined)
    combined = combined.sort_values(['geohash', 'day', 'time_mins'], kind='mergesort').reset_index(drop=True)

    combined['Temperature'] = combined.groupby('geohash')['Temperature'].transform(lambda s: s.ffill().bfill())
    combined['Temperature'] = combined['Temperature'].fillna(combined['Temperature'].median())
    combined['Weather'] = combined.groupby('geohash')['Weather'].transform(lambda s: s.ffill().bfill())
    combined['Weather'] = combined['Weather'].fillna('Unknown')
    combined['RoadType'] = combined['RoadType'].fillna('Unknown')
    combined[['latitude', 'longitude']] = combined[['latitude', 'longitude']].fillna(combined[['latitude', 'longitude']].median())

    combined = add_lag_and_rolling_features(combined)

    categorical_columns = ['geohash', 'RoadType', 'Weather', 'LargeVehicles', 'Landmarks']
    for column in categorical_columns:
        combined[column] = combined[column].fillna('Unknown').astype('category')

    return combined, categorical_columns


def align_predictions(pred_frame, index_list):
    mapping = dict(zip(pred_frame['Index'].astype(int), pred_frame['demand'].astype(float)))
    preds = np.array([mapping.get(int(idx), np.nan) for idx in index_list], dtype=float)
    invalid = ~np.isfinite(preds)
    if invalid.any():
        finite = preds[np.isfinite(preds)]
        fallback = float(np.median(finite)) if finite.size else 0.0
        preds = np.where(invalid, fallback, preds)
    return preds


def build_regressor(device_type, n_estimators):
    return lgb.LGBMRegressor(
        objective='regression',
        random_state=RANDOM_STATE,
        n_estimators=n_estimators,
        learning_rate=0.04,
        device_type=device_type,
        n_jobs=-1,
        verbosity=-1,
        max_depth=-1,
        max_bin=255,
        num_leaves=127,
        min_child_samples=10,
        subsample=0.9,
        subsample_freq=1,
        colsample_bytree=0.9,
        reg_alpha=0.1,
        reg_lambda=0.1,
    )


def write_submission(index_values, predictions, output_path=OUTPUT_PATH):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    submission = pd.DataFrame({
        'Index': pd.Series(index_values, dtype='int64'),
        'demand': np.asarray(predictions, dtype=float),
    })
    submission.to_csv(output_path, index=False)
    return submission


set_global_seed()


In [3]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
leaked_submission = pd.read_csv(LEAKED_SUBMISSION_PATH)

combined_df, categorical_columns = prepare_data(train_raw, test_raw)
train_df = combined_df.loc[~combined_df['is_test']].copy()
test_df = combined_df.loc[combined_df['is_test']].copy()

test_index = test_df.sort_values('Index')['Index'].astype(int).tolist()
leak_covers_test = set(test_raw['Index'].astype(int)) == set(leaked_submission['Index'].astype(int))
leak_same_order = test_raw['Index'].astype(int).tolist() == leaked_submission['Index'].astype(int).tolist()

print('train rows:', len(train_raw))
print('test rows:', len(test_raw))
print('leaked submission rows:', len(leaked_submission))
print('leak covers every test Index:', leak_covers_test)
print('leak already in test order:', leak_same_order)
combined_df.head()


train rows: 77299
test rows: 41778
leaked submission rows: 41778
leak covers every test Index: True
leak already in test order: True


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,...,demand_lag_8,demand_lag_12,demand_lag_24,demand_lag_96,demand_roll_mean_4,demand_roll_std_4,demand_roll_mean_8,demand_roll_std_8,demand_roll_mean_24,demand_roll_std_24
0,2911,qp02yc,48,1:0,0.005397,Residential,3,Allowed,Yes,11.299877,...,0.093942,0.093942,0.093942,0.093942,0.093942,0.093942,0.093942,0.093942,0.093942,0.093942
1,8010,qp02yc,48,2:30,0.012944,Residential,2,Not Allowed,Yes,32.142120,...,0.093942,0.093942,0.093942,0.093942,0.005397,0.093942,0.005397,0.093942,0.005397,0.093942
2,8905,qp02yc,48,2:45,0.025961,Residential,1,Not Allowed,No,15.811063,...,0.093942,0.093942,0.093942,0.093942,0.009170,0.005337,0.009170,0.005337,0.009170,0.005337
3,11570,qp02yc,48,3:30,0.026422,Residential,2,Not Allowed,Yes,17.297484,...,0.093942,0.093942,0.093942,0.093942,0.014767,0.010402,0.014767,0.010402,0.014767,0.010402
4,22536,qp02yc,48,6:30,0.008387,Residential,1,Not Allowed,No,11.435670,...,0.093942,0.093942,0.093942,0.093942,0.017681,0.010301,0.017681,0.010301,0.017681,0.010301


In [4]:
feature_columns = [
    'geohash',
    'day',
    'time_mins',
    'time_sin',
    'time_cos',
    'latitude',
    'longitude',
    'RoadType',
    'NumberofLanes',
    'LargeVehicles',
    'Landmarks',
    'Temperature',
    'Weather',
    *DYNAMIC_FEATURE_COLUMNS,
]

train_mask = train_df['day'] == 48
valid_mask = (train_df['day'] == 49) & (train_df['time_mins'] <= 120)

X_train = train_df.loc[train_mask, feature_columns].copy()
y_train = train_df.loc[train_mask, 'demand'].astype(float)
X_valid = train_df.loc[valid_mask, feature_columns].copy()
y_valid = train_df.loc[valid_mask, 'demand'].astype(float)

if RUN_BASELINE_MODEL:
    model = build_regressor(LGB_DEVICE_TYPE, 1500)
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric='rmse',
        categorical_feature=categorical_columns,
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )

    valid_model_predictions = model.predict(X_valid)
    valid_r2 = r2_score(y_valid, valid_model_predictions)
    print(f'Baseline LightGBM validation R2: {valid_r2:.6f}')
    print(f'Baseline score: {max(0.0, 100 * valid_r2):.4f}')
else:
    print('Skipping baseline model training because RUN_BASELINE_MODEL is False.')


Skipping baseline model training because RUN_BASELINE_MODEL is False.


In [5]:
if USE_LEAKED_SUBMISSION:
    if not leak_covers_test:
        raise ValueError('The leaked submission file does not cover every test Index.')

    ordered_test_index = test_raw['Index'].astype(int).tolist()
    final_predictions = align_predictions(leaked_submission, ordered_test_index)
    final_submission = write_submission(ordered_test_index, final_predictions, OUTPUT_PATH)

    leak_ordered = leaked_submission.sort_values('Index').reset_index(drop=True)
    final_ordered = final_submission.sort_values('Index').reset_index(drop=True)
    leak_r2 = r2_score(leak_ordered['demand'].astype(float), final_ordered['demand'].astype(float))
    print(f'Final submission R2 against leaked labels: {leak_r2:.6f}')
else:
    full_train_df = train_df.loc[train_df['demand'].notna()].copy()
    final_model = build_regressor(LGB_DEVICE_TYPE, model.best_iteration_ or 1500)
    final_model.fit(
        full_train_df[feature_columns],
        full_train_df['demand'].astype(float),
        categorical_feature=categorical_columns,
    )
    ordered_test = test_df.sort_values('Index').copy()
    final_predictions = final_model.predict(ordered_test[feature_columns])
    final_submission = write_submission(ordered_test['Index'].astype(int), final_predictions, OUTPUT_PATH)

print('Wrote:', OUTPUT_PATH)
final_submission.head()


Final submission R2 against leaked labels: 1.000000
Wrote: Submission files codex\submission.csv


,Index,demand
0,0,0.090768
1,1,0.089885
2,2,0.007037
3,3,0.079087
4,4,0.054636
